## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 02. KV 캐시 붙이기

**목표**: 01번에서 만든 모델에 KV 캐시를 붙이고, 캐시가 있을 때와 없을 때 속도 차이를 **내 맥에서 직접** 재 봐요.

사이트의 강의 "Prefill과 Decode, 그리고 KV 캐시"를 먼저 보고 오세요.

01번에서 만든 함수들(RMSNorm, RoPE, 어텐션, MLP)은 정답 버전을 불러와서 써요. 이번에 채울 건 **캐시를 쓰는 forward** 하나예요.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from common import *
from solutions.sol01 import apply_rope, attention, forward, greedy_generate, mlp, rms_norm, rope_cos_sin

setup_korean_font()
device = pick_device()
model, tok = load_model(device)
cfg, W = model_cfg(model), extract_weights(model)
ids = tok("The capital of France is", return_tensors="pt").input_ids[0].to(device)
print("준비 완료:", device)

## 1. 먼저 낭비를 재 보기

캐시 없이 토큰을 하나씩 만들면서, **스텝마다 걸린 시간**을 기록해요. 문장이 길어질수록 어떻게 될까요?

In [ ]:
@torch.no_grad()
def time_per_step_no_cache(tokens, n_new):
    times = []
    for _ in range(n_new):
        sync(device); t0 = time.perf_counter()
        logits = forward(tokens, W, cfg)              # 매번 전체를 다시 계산
        nxt = logits[-1].argmax()
        sync(device); times.append(time.perf_counter() - t0)
        tokens = torch.cat([tokens, nxt[None]])
    return times

no_cache_times = time_per_step_no_cache(ids, 120)
print(f"첫 스텝 {no_cache_times[1]*1000:.1f}ms → 마지막 스텝 {no_cache_times[-1]*1000:.1f}ms")

## 2. 캐시를 쓰는 forward 만들기

01번의 `forward`와 거의 같아요. 다른 점은 세 가지예요.

1. `tokens`는 **이번에 새로 넣는 토큰만**이에요. prefill 때는 프롬프트 전체, decode 때는 토큰 1개.
2. RoPE 위치는 0부터가 아니라 **`start_pos`부터** 시작해요: `torch.arange(start_pos, start_pos + T)`
3. 레이어마다 새로 만든 `k`, `v`를 **캐시 뒤에 붙이고**, 어텐션은 **캐시 전체**로 해요.
   ```
   c["k"] = k if c["k"] is None else torch.cat([c["k"], k])
   c["v"] = (같은 방식)
   a = attention(q, c["k"], c["v"])        ← 01번에서 S > T인 경우도 채점했던 이유!
   ```

캐시 구조는 레이어마다 `{"k": 텐서, "v": 텐서}` 하나씩, 30개짜리 리스트예요.

In [ ]:
def new_cache(cfg):
    return [{"k": None, "v": None} for _ in range(cfg["L"])]


def forward_with_cache(tokens, W, cfg, cache, start_pos):
    T = tokens.shape[0]
    x = W["embed"][tokens]
    ...  # 여기를 채우세요 (위치 → cos/sin → 레이어 30개 → norm → lm_head)


# 막히면:
# from solutions.sol02 import forward_with_cache

In [ ]:
# 채점 1: prefill 결과가 캐시 없는 forward와 같아야 해요.
# 채점 2: 그다음 토큰 하나를 decode한 결과가, 전체를 다시 계산한 결과의 마지막 줄과 같아야 해요.
with torch.no_grad():
    cache = new_cache(cfg)
    pre = run_check("prefill", lambda: check_close("prefill", forward_with_cache(ids, W, cfg, cache, 0), forward(ids, W, cfg), atol=1e-3))
    if pre:
        nxt = torch.tensor([1234], device=device)
        full = forward(torch.cat([ids, nxt]), W, cfg)[-1:]
        run_check("decode 1스텝", lambda: check_close("decode 1스텝", forward_with_cache(nxt, W, cfg, cache, ids.shape[0]), full, atol=1e-3))

## 3. 캐시로 문장 만들기

```
cache = new_cache(cfg)
logits = forward_with_cache(프롬프트 전체, ..., start_pos=0)        ← prefill
n_new번 반복:
    nxt = logits[-1].argmax()
    out 뒤에 붙이기
    logits = forward_with_cache(nxt[None], ..., start_pos=방금 붙인 토큰의 위치)   ← decode
```
방금 붙인 토큰의 위치는 `out.shape[0] - 1` 이에요.

In [ ]:
def generate_with_cache(tokens, W, cfg, n_new):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol02 import generate_with_cache

In [ ]:
with torch.no_grad():
    a = greedy_generate(ids, W, cfg, 20)
    b = generate_with_cache(ids, W, cfg, 20)
    run_check("캐시 생성 = 캐시 없는 생성", lambda: check_equal("캐시 생성 = 캐시 없는 생성", None if b is None else b.tolist(), a.tolist()))

## 4. 속도 비교 그래프

캐시를 쓸 때도 스텝마다 시간을 재서, 1번에서 잰 것과 같이 그려요.

In [ ]:
@torch.no_grad()
def time_per_step_cache(tokens, n_new):
    times, cache = [], new_cache(cfg)
    logits = forward_with_cache(tokens, W, cfg, cache, 0)
    out = tokens
    for _ in range(n_new):
        sync(device); t0 = time.perf_counter()
        nxt = logits[-1].argmax()
        out = torch.cat([out, nxt[None]])
        logits = forward_with_cache(nxt[None], W, cfg, cache, out.shape[0] - 1)
        sync(device); times.append(time.perf_counter() - t0)
    return times

cache_times = time_per_step_cache(ids, 120)
plt.figure(figsize=(8, 4))
plt.plot([t * 1000 for t in no_cache_times[1:]], label="캐시 없음")
plt.plot([t * 1000 for t in cache_times[1:]], label="KV 캐시")
plt.xlabel("생성한 토큰 수"); plt.ylabel("스텝당 시간 (ms)"); plt.legend(); plt.grid(alpha=.3)
plt.title("토큰 하나 만드는 데 걸리는 시간")
plt.show()
print(f"120토큰 총 시간: 캐시 없음 {sum(no_cache_times):.2f}초, KV 캐시 {sum(cache_times):.2f}초")

## 5. 캐시는 메모리를 얼마나 먹었나

공식: 토큰당 `2 × 레이어 × KV헤드 × head_dim × 바이트`. 지금은 float32라 4바이트예요.
실제 캐시 텐서의 크기와 공식이 같은지 확인해 보세요.

In [ ]:
cache = new_cache(cfg)
with torch.no_grad():
    forward_with_cache(ids, W, cfg, cache, 0)
real = sum(c["k"].numel() * c["k"].element_size() + c["v"].numel() * c["v"].element_size() for c in cache)
per_tok = 2 * cfg["L"] * cfg["nkv"] * cfg["hd"] * 4
print(f"토큰 {ids.shape[0]}개 → 실제 캐시 {real:,} 바이트")
print(f"공식: {per_tok:,} 바이트/토큰 × {ids.shape[0]} = {per_tok * ids.shape[0]:,} 바이트")

## 정리

- 캐시가 없으면 스텝 시간이 문장 길이에 따라 **계속 늘어나요**. 캐시가 있으면 거의 **일정**해요.
- 대신 캐시는 토큰마다 메모리를 차지해요. SmolLM2는 작아서 괜찮지만, Llama-8B는 토큰당 128KB(FP16)예요.

**생각해 보기**: 그래프에서 캐시를 쓸 때도 시간이 아주 조금씩 늘어난다면, 어떤 부분 때문일까요? (힌트: 어텐션은 캐시 전체를 읽어요.)

다음: **03_roofline_on_my_mac.ipynb**